# 04 - Receiver geometries: DRMBox, SurfaceGrid and PointCloud

ShakerMaker computes ground motion at a set of **receiver stations**. Several
helper classes build common receiver layouts:

- **DRMBox** - a closed box of stations (interior + exterior boundary) used for
  the Domain Reduction Method (DRM).
- **SurfaceGrid** - a regular grid in `'plane'`, `'hollow'` or `'filled'` mode.
- **PointCloudDRMReceiver** - stations read directly from an FEM node export (TSV).

For each one we do a simple 3D scatter of the station coordinates. The QA /
internal stations are marked differently. Every figure is saved as a `.png` in
this folder.

All coordinates are in **km**. We keep the layouts tiny (large spacing) so the


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from shakermaker.sl_extensions import DRMBox, SurfaceGrid, PointCloudDRMReceiver

## Helper: scatter the stations of a StationList

We iterate the station list, read each station's `.x` (xyz in km) and its
`is_internal` flag, then scatter exterior points in one color and


In [ ]:
def scatter_stations(stations, title, savename):
    ext, intr = [], []
    for s in stations:
        (intr if s.is_internal else ext).append(s.x)
    ext = np.array(ext).reshape(-1, 3)
    intr = np.array(intr).reshape(-1, 3)

    fig = plt.figure(figsize=(7, 6))
    ax = fig.add_subplot(111, projection='3d')
    if len(ext):
        ax.scatter(ext[:, 1], ext[:, 0], -ext[:, 2], c='tab:blue',
                   s=20, label='external')
    if len(intr):
        ax.scatter(intr[:, 1], intr[:, 0], -intr[:, 2], c='tab:red',
                   s=45, marker='^', label='internal / QA')
    ax.set_xlabel('(Y) Easting (km)')
    ax.set_ylabel('(X) Northing (km)')
    ax.set_zlabel('(Z) Depth (km)')
    ax.set_title(f'{title}  (nstations={stations.nstations})')
    ax.legend()
    plt.gcf().savefig(savename, dpi=150, bbox_inches='tight')
    return fig

## DRMBox

A small `1x1x1` box at center `[6, 8, 0]` km with 10 m spacing. The DRMBox
generates an interior boundary (`internal=True`), an exterior boundary


In [ ]:
drm = DRMBox([6.0, 8.0, 0.0], [1, 1, 1], [0.010, 0.010, 0.010],
             metadata={'name': 'small_drm'})
print('DRMBox stations:', drm.nstations)
scatter_stations(drm, 'DRMBox', 'drmbox_geometry.png')

## SurfaceGrid (plane)

An XY plane at `z = 0` with 2 km spacing. All grid points are exterior; the


In [ ]:
sg = SurfaceGrid([6.0, 8.0, 0.0], [5, 5, 0], [2.0, 2.0, 2.0],
                 mode='plane', plane_z=0.0, metadata={'name': 'plane_z0'})
print('SurfaceGrid stations:', sg.nstations)
scatter_stations(sg, 'SurfaceGrid plane (z=0)', 'surfacegrid_geometry.png')

## PointCloudDRMReceiver

Stations read from the tiny TSV `_drm_nodes.txt` (columns
`Node_ID  X  Y  Z  Type`, tab-separated). FEM coordinates (mm, Z-up) are
transformed to ShakerMaker coordinates (km, Z-down). Nodes flagged


In [ ]:
import os
nodes_file = os.path.join('..', '_drm_nodes.txt')
pc = PointCloudDRMReceiver(
    point_cloud_file=nodes_file,
    crd_scale=1 / 1e6,
    x0_fem=[22000., 15500., 0.],
    drmbox_x0=[6.0, 8.0, 0.0],
    metadata={'name': 'PointCloud_DRM'})
print('PointCloud stations:', pc.nstations)
scatter_stations(pc, 'PointCloudDRMReceiver', 'pointcloud_geometry.png')